# Supercooling in the Model Output

The degree of supercooling is given by

$$ \Delta T_{SC} = T_F - T $$


We distinguish between 
- **model potential supercooling**: with $T_F$ as the (constant) model freezing point -1.9
- **physical potential supercooling**: with $T=T_{pot}$ and $T_F = T_F(p=0,S)$ (physical surface-referenced freezing point)
- **in-situ supercooling**: with $T=T_{in-situ}$ and $T_F = T_F(p,S)$ (physical in-situ freezing point)

## Import Packages & Load Data

In [1]:
# packages for data processing
import numpy as np
import pandas as pd
import xarray as xr
from xmitgcm import open_mdsdataset
import gsw as gsw

# packages for plotting
import matplotlib.pyplot as plt
import plotly.graph_objects as go # interactive/3D plotting
import plotly.express as px
import cmocean.cm as cmo # ocean colormaps

# my own plotting functions
from plotting_functions import *

# technical packages
import warnings

In [2]:
model_run = "MSL004"
delta_t = 4 # time step in seconds
output_dir = "../../MITgcm/so_plumes/" + model_run

# surpress the xmitgcm warning about future changes with timedelta
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=FutureWarning)

    # open dataset
    ds = open_mdsdataset(output_dir, prefix = ['Eta', 'U', 'W', 'T', 'V', 'S', 'PH'], delta_t = delta_t, geometry = "cartesian")

## Model Potential Supercooling

To calculate the model potential supercooling we only need the (constant) model freezing point `Tfreezing` and the potential temperature, which is already included in the model output.

In [4]:
Tfreezing = -1.9
ds["model_SC"] = (Tfreezing-ds["T"])

## In-Situ Supercooling

To calculate the in-situ supercooling we need to first calculate
- the in situ temperature
- the in-situ freezing point

In-situ temperature:



In [8]:
# conservative temperature
ds["CT"] = gsw.CT_from_pt(ds["S"], ds["T"])
# calculate sea pressure ---- THIS IS ONLY PRELIMINARY AND  PROBABLY NOT CORRECT
ds["p"] = ds["PH"] + ds["PHrefC"]
ds["T_situ"]=gsw.t_from_CT(ds["S"], ds["CT"], ds["p"])

In-situ freezing point

`gsw.t_freezing(SA, p, saturation_fraction)`

compare with

`gsw.t_freezing_poly(SA, p, saturation_fraction)` (Polynomial)

In [9]:
# freezing point
ds["T_f"] = gsw.t_freezing(ds["S"], ds["p"], 0) # what about oxygen saturation fraction?

In [10]:
# in situ supercooling
ds["SC_situ"] = ds["T_f"] - ds["T_situ"]

## Plot Lead View

In [11]:
# crop to lead view
x_min = 0.0
x_max = 1190.0
mask = (ds["XC"] >= x_min) & (ds["XC"] <= x_max)
ds_sub = ds.where(mask, drop=True)

In [12]:
ds["SC_situ"].max().values

array(0.00808155)

In [13]:
print(ds["model_SC"].max().values)
print(ds["T"].min().values)

0.0010000467
-1.901


In [8]:
# monodirectional colormap (we set cmin=0 for this)
cube_time_evol(ds_sub, "model_SC", model_run, colorscale="ice_r", grid=False)

In [28]:
# diverging colormap (this forces cmin=cmax)
cube_time_evol(ds_sub, "model_SC", model_run, colorscale="balance_r", grid=False)